# 第 13 课 Notebook：Release Gate 与 FDE 作品答辩：从 Demo 到可交付系统

**核心问题：分数、风险和发布范围冲突时，团队怎样给出可辩护的决定？**

本 Notebook 先用本地固定夹具运行正常路径，再主动触发一条失败路径。所有副作用均为内存、临时目录或仿真；不需要 API Key、外网、真实邮箱、ERP、支付或机器人。代码字段与日志使用英文，讲解使用中文。

## 阅读路线

1. 先预测终态；2. 认识本课核心对象；3. 运行正常路径；4. 触发失败；5. 写出不变量；6. 把局部机制接到 Project。不要先背框架名称，先沿状态变化解释“事情为什么发生”。

## 1. 先预测

在运行前写下你的判断：如果系统只检查最终文字，不读取环境终态，本课案例最可能漏掉什么？课程核心问句是：**分数、风险和发布范围冲突时，团队怎样给出可辩护的决定？**

In [ ]:
lesson_id = 13
essential_question = '分数、风险和发布范围冲突时，团队怎样给出可辩护的决定？'
prediction = "write your prediction before running the next cell"
print({"lesson_id": lesson_id, "question": essential_question, "prediction": prediction})

## 2. 核心对象：`evaluate_project`

先看对象的边界：它接收什么、保存什么状态、返回什么终态、哪种失败必须由外部程序或人处理。下面先使用本课公开的教学核心，观察接口和状态；完整实现留在项目 starter 的任务边界中。

In [ ]:
print({"teaching_boundary": "本页只观察公开教学核心的输入、状态和输出；不读取教师实现源码。"})

## 3. 正常路径：先看真实结果

运行时先找输入、关键中间状态和终态。若输出很长，不要逐行朗读；只标出本课负责的新对象。

In [ ]:
from agent_course.miniprojects.release_gate import run_demo

release_summary = run_demo()
print(release_summary)

## 4. 解释正常路径

用四句话复述：输入是什么；谁拥有状态；程序允许了哪个动作；哪个环境结果证明任务完成。若只能复述 Agent 的自然语言，就还没有解释闭环。

In [ ]:
normal_path_check = {
    "input_visible": True,
    "state_visible": True,
    "terminal_outcome_visible": True,
    "real_external_side_effect": False,
}
required_checks = (
    "input_visible",
    "state_visible",
    "terminal_outcome_visible",
)
assert all(normal_path_check[key] for key in required_checks)
assert normal_path_check["real_external_side_effect"] is False
print(normal_path_check)

## 5. 失败路径：能力必须和边界一起出现

现在保持任务与实现不变，只改变一个关键条件。观察失败发生在提议、授权、执行、反馈还是终态判断。

In [ ]:
decisions = {item["project"]: item for item in release_summary["decisions"]}
assert decisions["warehouse"]["decision"].value == "blocked"
assert decisions["code"]["decision"].value == "limited"
assert decisions["research"]["decision"].value == "release"

## 6. 把正常与失败路径画在一起

下面的图直接读取本次运行产生的状态、结果与停止原因。先沿图复述“输入—动作—环境结果—终态”，再回到代码定位每一项由哪一行产生。图不是额外的装饰层。

In [ ]:
from IPython.display import HTML, display

from agent_course.courseware.project_visuals import (
    ReleaseRow,
    render_release_matrix,
)

release_rows = tuple(
    ReleaseRow(
        project=item["project"],
        decision=item["decision"].value,
        tested_scope=tuple(item["tested_scope"]),
        reasons=tuple(item["reasons"]),
        canary_percent=item["canary_percent"],
    )
    for item in release_summary["decisions"]
)
display(
    HTML(
        render_release_matrix(
            title="所选轨道的 Release Gate",
            question="平均分不错时，哪一个硬门仍会阻止发布？",
            rows=release_rows,
            takeaway="硬门先于均值；每个决定同时写明范围、canary 和 rollback。",
        )
    )
)

## 6.1 不变量

不变量是无论模型怎样表达都不能被绕过的规则。它应由普通代码、存储约束、沙箱或独立监督器执行。

In [ ]:
invariants = (
    "terminal outcome is explicit",
    "failure is represented as data",
    "side effects stay inside the course sandbox",
    "a model statement cannot override a deterministic boundary",
)
assert len(invariants) == 4
print(invariants)

## 7. 连接到 Project

发布答辩：任务集、重复试验、硬门、限量发布、Canary、Rollback 与 incident-to-eval。 本节代码不是另一本工具说明书，而是项目验收中的一个可重复证据。

## 8. 学生练习

完成下面的小函数。允许使用 Agent 帮助阅读报错和补充测试，但你必须能逐行解释判断条件，并亲自确认失败路径。

In [ ]:
def release_allowed(hard_failures: list[str], grouped_pass: bool) -> bool:
    # TODO: 硬门失败不能被平均分抵消。
    raise NotImplementedError

assert release_allowed([], True)
assert not release_allowed(["unsafe_stop"], True)

## 9. 展示与验收

展示时只讲五件事：任务输入、关键状态、正常终态、失败终态、一个不能被模型绕过的不变量。练习时改变一项夹具，解释结果为什么改变。

## 10. 小结

回答开场问题：**分数、风险和发布范围冲突时，团队怎样给出可辩护的决定？**

请用“输入 → 状态 → 动作 → 环境结果 → 停止原因”写一段不超过 120 字的复盘，并把失败断言迁移到 Project 测试。